Groundwater | Flow Modeling Track

# Step 9: Documentation & Maintenance — Making Your Model Reproducible

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → **9-Document** → 10-Communicate

**Story so far:** You have a calibrated MODFLOW 6 model of the Limmat Valley (Step 5), cross-validated it (Step 6), quantified parameter uncertainty (Step 7), and applied it to delineate protection zones for a proposed pumping well (Step 8). But could a colleague — or future you — reproduce any of this six months from now?

| **Reading time** | ~15 minutes |
|:--|:--|

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Identify** the essential components of model documentation
2. **Write** a model metadata file, decision log, and limitations statement
3. **Apply** version-control best practices to model files
4. **Prepare** a model archive that others can reproduce

---

## Prerequisites

- **Completed Notebooks 1–8** — you need the full modelling experience to appreciate why documentation matters

---

## 1 — Why Documentation Matters

Groundwater models outlive their creators' memory. Without documentation, common questions become unanswerable:

- *"What did we assume for recharge?"* — No record of the 99 mm/year value or its derivation
- *"Which version was used for the protection zone report?"* — Multiple folders, unclear which is final
- *"Why single-layer?"* — The geological reasoning is lost
- *"Can we update with new wells?"* — The calibration workflow is unclear

Systematic documentation serves three audiences:

| Audience | Needs | Example from our model |
|----------|-------|----------------------|
| **Future you** | Reproduce results, remember decisions | Why did we use a variogram range of 6000 m? |
| **Colleagues** | Understand and modify the model | How to add a new observation well and recalibrate |
| **Stakeholders** | Trust the results | What are the model's limitations for regulatory use? |

Professional standards require it: the **Australian Groundwater Modelling Guidelines** (Barnett et al., 2012), **ASTM D5718**, and **USGS model archiving standards** all mandate comprehensive documentation.

---

## 2 — What to Document

Five components form the backbone of model documentation. Here they are, illustrated with our Limmat Valley model.

### 2.1 Model Metadata

Basic identifying information in a structured, machine-readable format:

```yaml
model_name: Limmat Valley Aquifer Model
version: 1.0.0
date_created: 2025-02-01
author: Your Name
organization: ETH Zürich
purpose: Educational – steady-state flow, calibration, scenario analysis
software: MODFLOW 6 (v6.5) via FloPy
grid: DISV Voronoi, ~5230 cells, single layer
crs: EPSG:2056 (Swiss LV95)
status: Calibrated (9 obs), cross-validated (leave-one-out), not independently validated
```

### 2.2 Decision Log

Record **why**, not just what. These decisions are invisible in the model files but critical for interpretation:

| Step | Decision | Rationale |
|------|----------|-----------|
| Notebook 2 | Single-layer model | Alluvial gravel is relatively homogeneous vertically; borehole logs show one productive unit |
| Notebook 4 | Voronoi (DISV) grid, ~50 m | Flexible geometry for irregular valley boundary; 50 m balances resolution and run time |
| Notebook 5 | 9 pilot points for K | 4 real AWEL wells + 5 synthetic observations; more pilot points would overfit |
| Notebook 5 | Variogram range = 6000 m, anisotropy 3:1 at −45° | Reflects elongated valley geometry (NW–SE) |
| Notebook 8 | Grid refinement to 10 m near well | S2 zone (~20–30 m) requires finer resolution than base 50 m cells |

### 2.3 Parameter Summary

Calibrated values with their justification and plausible ranges:

| Parameter | Calibrated | Unit | Range | Source |
|-----------|-----------|------|-------|--------|
| K (alluvial gravel) | ~5–50 (spatially variable) | m/d | 1–100 | Pilot-point calibration; literature for Alpine gravels |
| Recharge | 99 | mm/yr | 70–150 | ~10% of precipitation (NB4) |
| River conductance (Limmat) | 100 | m²/d | 10–500 | Estimated from riverbed properties |
| Sihl leakance | calibrated | 1/d | — | Adjusted via PEST++ (NB5) |
| Porosity | 0.15 | — | 0.10–0.25 | Literature value for PRT (NB8) |

### 2.4 Data Sources

| Data | Source | Retrieved | Quality |
|------|--------|-----------|---------|
| Topography / aquifer base | Swisstopo DHM25, borehole logs | 2024 | 25 m resolution |
| Model boundary | Geological Atlas 1:25 000 | 2024 | Manually digitised |
| Groundwater levels | AWEL monitoring network | 2024 | 4 wells, mixed CRS (LV03/LV95) |
| River geometry | Canton Zürich open geodata | 2024 | Limmat + Sihl |

### 2.5 Limitations Statement

Be explicit about what the model **cannot** do. This protects both you and the decision-maker. It informs about appropriate uses: screening-level analysis, educational demonstration, scenario comparison; but also inappropriate uses: regulatory decisions without additional validation, quantitative contaminant transport predictions.

| Limitation | Consequence |
|-----------|-------------|
| **Steady-state only** | Cannot predict response times or transient recovery |
| **Single layer** | No vertical gradients — unsuitable for multi-aquifer problems |
| **4 real observation wells** | Calibration is under-constrained; cross-validation shows ~1 m RMSE |
| **No transport** | Flow paths via PRT, but no concentrations or dispersion |
| **Unvalidated in western domain** | Protection zone results (NB8) carry high uncertainty there |



---

## 3 — Version Control with Git

Version control tracks every change to your model, enabling rollback, collaboration, and a complete audit trail. Git is the standard.

### 3.1 - What to Track vs. Ignore

| Track (text/small files) | Ignore (large/binary) |
|--------------------------|----------------------|
| Input files (.nam, .dis, .npf, etc.) | Output files (.hds, .cbc, .lst) |
| Scripts (.py, .ipynb) | Large binary datasets |
| Documentation (.md, .yaml) | Temporary / scratch files |
| Configuration and environment files | External data (link to source instead) |

**Tip:** Add a `.gitignore` with `*.hds`, `*.cbc`, `*.lst`, `__pycache__/`, `.ipynb_checkpoints/`.

### 3.2 - Commit Messages That Help

Good messages explain **what** changed and **why**:

```
Good:  "Increase K in zone 2 from 10 to 25 m/d based on pump test KS-2024-03"
Good:  "Add Sihl as RIV boundary — gauge data 2015-2020, improves RMSE from 1.4 to 0.8 m"

Bad:   "Updated model"
Bad:   "Changes"
```

### 3.3 - Versioning Scheme

Use **semantic versioning** (MAJOR.MINOR.PATCH) for model releases:

| Change | Version bump | Example |
|--------|-------------|---------|
| New layer, new boundaries | **MAJOR** (1.0 → 2.0) | Add confining unit |
| Recalibration, new stress | **MINOR** (1.0 → 1.1) | Add pumping well from NB8 |
| Bug fix, typo correction | **PATCH** (1.0.0 → 1.0.1) | Fix inactive-cell WEL entries |

Tag releases in Git: `git tag -a v1.1.0 -m "Add proposed pumping well scenario"`

---

## 4 — Archiving for Reproducibility

A model archive is a self-contained package that lets someone else reproduce your results. Structure it clearly:

```
limmat_valley_v1.0.0/
├── README.md                  # Overview, quick-start instructions, contact
├── model_metadata.yaml        # Version, author, purpose, status
├── model_files/               # MODFLOW 6 input files
│   ├── mfsim.nam
│   ├── limmat_valley.dis
│   └── ...
├── scripts/                   # FloPy build & post-processing scripts
│   ├── build_model.py
│   └── run_scenarios.py
├── data/                      # Input data (or links to sources)
│   ├── observations.csv
│   └── boundary_data/
├── documentation/             # Decision log, parameter table, limitations
│   └── decision_log.md
├── pyproject.toml             # Python dependencies (or environment.yml)
└── .gitignore
```

### 4.1 - Where to Archive

| Location | Use case | Retention |
|----------|----------|-----------|
| **GitHub / GitLab** | Active development, collaboration | Ongoing |
| **Zenodo / Figshare** | Citable archive with DOI | Permanent |
| **Institutional server** | Official project archives | 10+ years |

### 4.2 - Reproducibility Checklist

Before archiving, verify:

1. Can someone clone the repository and run the model without contacting you?
2. Are all software dependencies pinned (e.g., `flopy==3.8.0`, `modflow6==6.5.0`)?
3. Are data sources documented — or included if small enough?
4. Is there a README with step-by-step instructions?
5. Does the archive include the specific model version used for each report or publication?

---

## 5 — Review and Quality Assurance

A model should not be released without review. Professional guidelines recommend a staged review process:

| Level | Who | What they check | When |
|---|---|---|---|
| **Self-check** | Modeller | Mass balance, plausibility, code review | After each major step |
| **Peer review** | Colleague modeller | Assumptions, calibration quality, documentation | Before reporting |
| **Independent review** | External specialist | Full methodology, data, results, conclusions | Before regulatory submission |

> **What we've already done:**
> Throughout this course, you've applied quality assurance at multiple stages:
> - **Verification:** Water balance checks in NB5 and NB6
> - **Cross-validation:** Leave-one-out testing in NB6
> - **Sensitivity analysis:** FOSM parameter screening in NB7
> - **Physical plausibility:** K-field and head-distribution checks in NB6
>
> These are all forms of self-check. A professional project would add formal peer and independent review.

### 5.1 Staged Reporting at Hold-Points

The Australian Groundwater Modelling Guidelines recommend formal review at three hold-points during a modelling project:

1. **After conceptualisation** — Is the perceptual model defensible? Are alternative conceptual models considered? (NB2)
2. **After calibration** — Does the model fit the data? Are parameters plausible? Is the calibration well-posed? (NB5–NB6)
3. **After prediction** — Are scenario results reasonable? Have uncertainties been communicated? (NB7–NB8)

At each hold-point, the modeller presents progress to the client or review panel, who can request changes before proceeding. This prevents costly rework and ensures the model development stays aligned with the project objectives.

---

## 6 — Summary Checklist

### Every Model Should Have

| Item | Purpose |
|------|---------|
| README.md | Overview, quick start, contact info |
| Model metadata (.yaml) | Version, date, author, software, status |
| Decision log | Rationale for key modelling choices |
| Parameter summary | Calibrated values with ranges and sources |
| Data source table | Where each dataset came from and when |
| Limitations statement | What the model cannot do |
| Version control (Git) | Change history and audit trail |

### Professional Projects Should Add

| Item | Purpose |
|------|---------|
| Model report (PDF) | Comprehensive technical documentation |
| Quality assurance record | Review and approval history |
| Update protocol | When and how to recalibrate |
| Archive with DOI | Citable, permanent, reproducible |

---

## What You're Taking Forward

| From this notebook | Used in |
|-------------------|---------|
| Documentation components (metadata, decision log, parameters, limitations) | Group case study — document your own model |
| Version control practices | All future modelling work |
| Archiving and reproducibility | Project submissions, publications |

---

## Next Steps

**Continue to:** [10_communication.ipynb](10_communication.ipynb) — where we learn to communicate model results effectively to different audiences.

---

## Key Messages

1. **Document as you go** — reconstructing decisions later is unreliable
2. **Record the *why*** — model files show *what*; only a decision log captures *why*
3. **State limitations prominently** — transparency builds trust
4. **Use version control** — Git is the professional standard
5. **Archive complete packages** — everything needed to reproduce results

---

## References

- Barnett, B. et al. (2012). *Australian Groundwater Modelling Guidelines*. National Water Commission. [PDF](https://www.epa.govt.nz/assets/FileAPI/proposal/NSP000028/Hearings-Week-01/c2cd52352e/02-Australian-Groundwater-modelling-guidelines.pdf)
- ASTM D5718-13 (2013). *Standard Guide for Documenting a Groundwater Flow Model Application*.
- Wilkinson, M.D. et al. (2016). The FAIR Guiding Principles for scientific data management. [*Nature Scientific Data*](https://www.nature.com/articles/sdata201618).